In this notebook I'll be using NTSB api to get aviation data.

In [29]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import json

In [30]:
load_dotenv()
API_KEY = os.environ["API_KEY"]

In [31]:
headers = {
    "Ocp-Apim-Subscription-Key": API_KEY,
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

In [32]:
url="https://api.ntsb.gov/public/api/Common/v2/GetCasesByDateRange/"

In [35]:
parameters = {
    "startDate": "01/01/2021",
    "endDate": "12/01/2025", 
    "mode":"aviation"
}

In [36]:
all_cases = []
marker = None
page = 0

while True:
    params = dict(parameters)
    if marker:
        params["marker"] = marker

    response = requests.get(url, headers=headers, params=params, timeout=300)
    if response.status_code == 204: 
        break
    response.raise_for_status()

    response_json = response.json()
    page += 1
    all_cases.extend(response_json["data"])
    print(f"Page {page}: {len(response_json['data'])} cases, "
          f"last event date {response_json['data'][-1]['eventDate'] if response_json['data'] else '-'}, "
          f"total {len(all_cases)}")

    if not response_json.get("hasMore"):
        break
    marker = response_json["nextMarker"]

Page 1: 1000 cases, last event date 2021-08-07, total 1000
Page 2: 1000 cases, last event date 2022-04-03, total 2000
Page 3: 1000 cases, last event date 2022-10-03, total 3000
Page 4: 1000 cases, last event date 2023-06-05, total 4000
Page 5: 1000 cases, last event date 2023-12-23, total 5000
Page 6: 1000 cases, last event date 2024-07-26, total 6000
Page 7: 1000 cases, last event date 2025-03-18, total 7000
Page 8: 1000 cases, last event date 2025-09-27, total 8000
Page 9: 270 cases, last event date 2025-12-01, total 8270


In [37]:
cases = list({case["mKey"]: case for case in all_cases}.values())
print("Downloaded:", len(all_cases), "| unique cases:", len(cases))

Downloaded: 8270 | unique cases: 8270


In [38]:
with open("ntsb_cases.json", "w", encoding="utf-8") as f:
    json.dump(cases, f, indent=2, ensure_ascii=False)

In [39]:
df = pd.json_normalize(cases)


In [40]:
print("Number of cases:", len(df))
display(df.head())

Number of cases: 8270


,mKey,isPost2008,isPre1982,ntsbNumber,completionStatus,hasSafetyRecommendation,leadAgency,boardMemberLaunch,boardMeetingDate,docketOpenDate,...,onboardSerious,onboardMinor,onboardNone,topicMode,investigatingParties,weatherConditions,weatherPrecipitations,aircrafts,narratives,richNarratives
0,102484,True,False,WPR21LA077,Completed,False,NTSB,False,NaN,NaN,...,0,1,0,Aviation,[],"[{'pilotWeatherSource': 'Automated report', 'a...",[],"[{'aircraftNumber': 1, 'aircraftCategory': 'He...","[{'aircraftNumber': 1, 'probableCause': 'The p...",None
1,102492,True,False,WPR21LA079,Completed,False,NTSB,False,NaN,NaN,...,0,0,4,Aviation,[],[{'pilotWeatherSource': 'National weather serv...,"[{'entryNumber': 1, 'descriptor': 'None', 'int...","[{'aircraftNumber': 1, 'aircraftCategory': 'Ai...","[{'aircraftNumber': 1, 'probableCause': 'A fai...",None
2,102503,True,False,WPR21WA081,N/A,False,Other,False,NaN,NaN,...,0,2,0,Aviation,[],"[{'pilotWeatherSource': None, 'accidentSiteCon...",[],"[{'aircraftNumber': 1, 'aircraftCategory': 'He...",[],None
3,102480,True,False,CEN21LA104,Completed,False,NTSB,False,NaN,NaN,...,0,0,0,Aviation,"[Lycoming, FAA, Piper Aircraft, Inc]","[{'pilotWeatherSource': 'None', 'accidentSiteC...",[],"[{'aircraftNumber': 1, 'aircraftCategory': 'Ai...","[{'aircraftNumber': 1, 'probableCause': 'The p...",None
4,102485,True,False,WPR21LA078,Completed,False,NTSB,False,NaN,NaN,...,0,0,4,Aviation,"[Vitatoe Aviation LLC, FAA-FSDO]","[{'pilotWeatherSource': 'TV/Radio weather', 'a...",[],"[{'aircraftNumber': 1, 'aircraftCategory': 'Ai...","[{'aircraftNumber': 1, 'probableCause': 'The p...",None


In [41]:
df.columns

Index(['mKey', 'isPost2008', 'isPre1982', 'ntsbNumber', 'completionStatus',
       'hasSafetyRecommendation', 'leadAgency', 'boardMemberLaunch',
       'boardMeetingDate', 'docketOpenDate', 'launchType', 'reportType',
       'reportNumber', 'reportDate', 'eventType', 'highestInjuryLevel',
       'investigationClass', 'factualFinalReportFlavor',
       'originalPublishedDate', 'publishDateTime', 'publishProfile',
       'investigationPage', 'docketPage', 'reportPage', 'caseClosed',
       'caseCreatedDateTimeUtc', 'factualPublishedDate', 'isStudy',
       'lastRevisionDate', 'mode', 'eventDate', 'eventTimeUtc',
       'eventTimeUtcOffsetHours', 'eventIsUtc', 'eventCity',
       'eventStateOrRegion', 'eventZipcode', 'eventCountry', 'eventLocation',
       'eventLatLongEstimated', 'eventLatitude', 'eventLongitude',
       'eventPropertyDamage', 'eventHazmatInvolved',
       'eventCollisionWithOtherAircraft', 'eventEnvironmentalDamage',
       'caseType', 'region', 'totalFatal', 'totalSeri